# Week 2 - In Class Activity: End-to-end Machine Learning Pipeline

**Goal:** Create and end-to-end pipeline for machine learning regression models.

In this workbook, you will use an **abalone** dataset. The age of abalone is determined by cutting the shell through the cone, staining it, and counting the number of rings through a microscope.  We will try to used other measurements, which are easier to obtain, to predict the age. 

By the end of the activity, you should be able to explain why:

1. Random k-fold cross-validation can leak information when nearby points or nearby months appear in both training and test folds.
2. Temporal block cross-validation better tests whether a model generalizes to unseen years.
3. Spatial block cross-validation better tests whether a model generalizes to unseen regions.
4. Blocked evaluation often gives worse, but more realistic, estimates of predictive performance.

**Dataset:**  Warwick J Nash, Tracy L Sellers, Simon R Talbot, Andrew J Cawthorn and
Wes B Ford (1994) "The Population Biology of Abalone (_Haliotis_
species) in Tasmania. I. Blacklip Abalone (_H. rubra_) from the North
Coast and Islands of Bass Strait", Sea Fisheries Division, Technical
Report No. 48 (ISSN 1034-3288)


## 1. Setup

This notebook intentionally gives you some complete code and some incomplete code. Sections marked **TODO** are places where you should write or modify code yourself.

In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.inspection import permutation_importance
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import matplotlib.pyplot as plt

## 2. Load the data

In [ ]:
dataset = pd.read_csv(
    "https://raw.githubusercontent.com/dblaskey/ML_Course_Code/main/Data/abalone.csv",
    header=None,
    names=["sex", "length", "diameter", "height", "whole_weight", "shucked_weight", "viscera_weight", "shell_weight", "rings"]
)
dataset

## 3. Data exploration

Before fitting a model, we explore the dataset so we understand what each variable represents and what data types are there.

In [ ]:
display(dataset.describe(include="all").T)

In [ ]:
# Determine the number of missing values
dataset.isna().sum().sort_values(ascending=False)

In [ ]:
# Determine the column types
dataset.dtypes

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(
    dataset["rings"],
    bins=range(dataset["rings"].min(), dataset["rings"].max() + 2),
    edgecolor="black"
)

plt.xlabel("Number of Rings")
plt.ylabel("Frequency")
plt.title("Histogram of Abalone Rings")

plt.show()

## 4. Model preprocessing and regression

We now build a simple model for number of abalone rings.

The response variable is `rings`.
    
The predictors are:

	Name		Data Type	Meas.	Description
	----		---------	-----	-----------
	Sex		nominal			M, F, and I (infant)
	Length		continuous	mm	Longest shell measurement
	Diameter	continuous	mm	perpendicular to length
	Height		continuous	mm	with meat in shell
	Whole weight	continuous	grams	whole abalone
	Shucked weight	continuous	grams	weight of meat
	Viscera weight	continuous	grams	gut weight (after bleeding)
	Shell weight	continuous	grams	after being dried

In [ ]:
# Create copy of dataset
df = dataset.copy()

In [ ]:
# Separate predictors and target
X = df.drop(columns=["rings"])
y = df["rings"]

### Feature Engineering

This creates new features based on other features. We will talk more about this next week.

In [ ]:
eps = 1e-6 # Prevent infinity

# Shape proxies
X["volume_proxy"] = X["length"] * X["diameter"] * X["height"]

# Composition ratios
X["shell_to_whole"] = X["shell_weight"] / (X["whole_weight"] + eps)
X["weight_per_size"] = X["whole_weight"] / (X["volume_proxy"] + eps)

# Density-like and residual-weight features
X["nonmeat_weight"] = X["whole_weight"] - X["shucked_weight"] - X["viscera_weight"]

X.head()

### Build the `scikit-learn` preprocessing pipeline
The pipeline separates numeric and categorical variables.

For the numeric variables, the workflow imputes missing values (although there are none here) with the median and standardizes the values.

For categorical variables, the workflow fills missing values with the most common category and then one-hot encodes the categories. `drop="first"` removes one category from each categorical variable so the regression has a reference category.

We are using a `Random Forest` model, but don't worry too much about the details to this model or its hyperparameters. We will talk more about it in a couple of weeks. Right now we are using it just for demonstration of the end-to-end machine learning pipeline.

In [ ]:
numeric_features = X.drop(columns = ["sex"]).columns
categorical_features = ["sex"]


# For now we will use median and most_frequent imputation, but we know that isn't always the best.
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop"
)

# Model
rf = RandomForestRegressor(criterion="poisson", random_state=42)

model_cv = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", rf)
])

# Hyperparameter grid
param_grid = {
    "model__n_estimators": [300, 500],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_split": [5, 10],
    "model__min_samples_leaf": [4, 8],
    "model__max_features": [1.0]
}
# Note you can add many more here, but the more hyper parameters you add, the longer it will take.

# Stratifying continuous targets directly is not possible, so use quantile bins.
# This helps keep rare high-ring observations represented in both partitions.
ring_bins = pd.qcut(y, q=10, labels=False, duplicates="drop")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=ring_bins
)

### Cross-validation

Cross-validation gives a more realistic estimate of predictive performance. The dataset is split into five parts. The model is trained on four parts and tested on the remaining part, repeated five times. All the information you could ever want to know about sklearn's cross validation can be found here: https://scikit-learn.org/stable/modules/cross_validation.html.  

One thing you may notice is that we set `scoring` to *neg_root_mean_squared_error*. We do this because sklearn requires scoring metrics to follow a "greater is better" rule, meaning higher scores must always represent better models.

In [ ]:
'''
WARNING: This cell will take a few minutes to run. While you are waiting feel free to read up on cross validation in the link above.
'''

cv = KFold(
    n_splits=5, # Number of splits you want to take
    shuffle=True, # Should the order of data be shuffled (not good for time series data)
    random_state=42
)

# Grid search
grid_search = GridSearchCV(
    estimator=model_cv, # The preprocess pipeline strategy created above
    param_grid=param_grid, # The parameter grid created above
    cv=cv, # The CV strategy created above
    scoring="neg_root_mean_squared_error",
    n_jobs=2, # You can run the process in parallel based on your number of processors
    verbose=1 # How much information is printed out each fold 
)

# Fit cross-validated model
grid_search.fit(X_train, y_train)

# Best model
best_model = grid_search.best_estimator_

print("Best parameters:")
print(grid_search.best_params_)

print("Best CV RMSE:")
print(-grid_search.best_score_)

# Test-set predictions
y_pred = best_model.predict(X_test)

# Test-set evaluation
mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2 = r2_score(y_test, y_pred)

print("\nTest set performance:")
print("Mean Absolute Error:", mae)
print("Root Mean Squared Error:", rmse)
print("R² Score:", r2)

In [ ]:
plt.figure(dpi=300)

plt.scatter(y_test, y_pred, alpha=0.6)

min_value = min(y_test.min(), y_pred.min())
max_value = max(y_test.max(), y_pred.max())

plt.plot(
    [min_value, max_value],
    [min_value, max_value],
    linestyle="--"
)

plt.xlabel("Actual Rings")
plt.ylabel("Predicted Rings")
plt.title("Predicted vs. Actual Rings")

plt.text(
    min_value,
    max_value,
    f"MAE = {mae:.2f}\nRMSE = {rmse:.2f}\nR² = {r2:.2f}",
    verticalalignment="top"
)

plt.show()

In [ ]:
results = pd.DataFrame({
    "actual": y_test,
    "predicted": y_pred,
})
results["residual"] = results["actual"] - results["predicted"]

plt.figure(dpi = 300)
plt.scatter(results["actual"], results["residual"], alpha=0.55)
plt.axhline(0, linestyle="--")
plt.xlabel("Actual rings")
plt.ylabel("Residual (actual - predicted)")
plt.title("Residuals: Positive Values Indicate Underprediction")
plt.show()

### Feature Importance

This is an indication of what provided the machine learning model the most useful information during training. We will talk about in more detail later in the semester.

In [ ]:
perm = permutation_importance(
    best_model,
    X_test,
    y_test,
    n_repeats=15,
    random_state=42,
    n_jobs=-1
)

importance = pd.DataFrame({
    "feature": X_test.columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=True)

display(importance.sort_values("importance_mean", ascending=False).head(15))

plt.figure(figsize=(9, 7))
plt.barh(
    importance["feature"],
    importance["importance_mean"],
    xerr=importance["importance_std"],
)
plt.xlabel("Decrease in score after permutation")
plt.ylabel("Feature")
plt.title("Permutation Importance")
plt.show()

## 5. Reflection questions

1. What are the required steps for the machine learning pipeline?
2. What test/train split was used for this model?
3. How is the model evaluated?
4. Did the model perform well? 

## 6. Change the pipeline

**TODO:** Update the pipeline to run a KNN approach instead of a random forest. For help on this problem, refer to sklearn's documentation of KNN: https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsRegressor.html

In [ ]:
#TODO